[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C27_Model_Compression_Course/05_pruning/05_pruning.ipynb)

# 05 · 剪枝与稀疏（用 numpy 从零实现）

把既有模型的权重删成 0，**从零实现幅度剪枝/2:4/微调恢复/SparseGPT，并验证**。

**路线**：
1. 幅度剪枝：删最小 |w|，精确命中目标稀疏度
2. 全局 vs 逐层剪枝：哪个更优
3. 2:4 半结构稀疏：每 4 个保留 2 个
4. 稀疏度-精度曲线：剪越狠掉越多
5. 剪枝-微调恢复：保留权重补偿被删的
6. **SparseGPT**：逆 Hessian 一次性重建，优于幅度剪枝
7. ✏️ 练习（全局/逐层剪枝 / 2:4 / 稀疏度-精度 / SparseGPT）
8. 📖 答案 · 🧪 真实 GPT-2 权重胶囊

> **本课纪律**：每个机制都对拍/验证。稀疏度精确、2:4 每组恰 2 非零、微调降误差、SparseGPT<幅度剪枝。

## 1 · 幅度剪枝：删最小 |w|

给定目标稀疏度 `s`，删掉 `|w|` 最小的 `s` 比例权重。用 `np.partition`(O(n) 部分排序)找阈值。
验证：实际稀疏度精确命中目标，且删的确实是最小的那些。

In [ ]:
import numpy as np
rng = np.random.default_rng(0)

def magnitude_prune(W, sparsity):
    '''按 |w| 删最小的 sparsity 比例, 返回 (剪枝后权重, 0/1 掩码)。'''
    if sparsity <= 0:
        return W.copy(), np.ones_like(W, dtype=bool)
    k = int(round(sparsity * W.size))                 # 要删的个数
    if k >= W.size:
        return np.zeros_like(W), np.zeros_like(W, dtype=bool)
    thresh = np.partition(np.abs(W).ravel(), k-1)[k-1]   # 第 k 小的 |w|
    mask = np.abs(W) > thresh
    return W * mask, mask

W = rng.standard_normal((32, 64)) * 0.1
print(f"{'目标稀疏度':>10} {'实际稀疏度':>10}")
for sp in [0.0, 0.3, 0.5, 0.9]:
    Wp, mask = magnitude_prune(W, sp)
    actual = (Wp == 0).mean()
    print(f'{sp:>10.1f} {actual:>10.2f}')
    assert abs(actual - sp) < 0.02, f'稀疏度应命中目标: {actual} vs {sp}'
# 删的应是最小的: 剪枝后最小的非零 |w| >= 被删的最大 |w|
Wp, mask = magnitude_prune(W, 0.5)
assert np.abs(Wp[mask]).min() >= np.abs(W[~mask]).max() - 1e-9, '应删最小的权重'
print('✅ 幅度剪枝正确：稀疏度精确命中，删的确实是 |w| 最小的')

## 2 · 全局 vs 逐层剪枝

**全局**：用一个统一阈值跨所有层比 `|w|`，各层稀疏度自适应。
**逐层**：每层各剪到同一比例。全局通常更优——不同层冗余度不同，全局让冗余层多剪、关键层少剪。

In [ ]:
# 两个量级差异很大的层(层A权重大、层B权重小)
WA = rng.standard_normal((16, 32)) * 1.0      # 大权重层
WB = rng.standard_normal((16, 32)) * 0.05     # 小权重层

def global_prune(weights, sparsity):
    '''全局: 把所有层权重拼一起定一个阈值。'''
    all_w = np.concatenate([np.abs(w).ravel() for w in weights])
    k = int(round(sparsity * all_w.size))
    thresh = np.partition(all_w, k-1)[k-1]
    return [w * (np.abs(w) > thresh) for w in weights]

def layerwise_prune(weights, sparsity):
    '''逐层: 每层各剪到 sparsity。'''
    return [magnitude_prune(w, sparsity)[0] for w in weights]

g = global_prune([WA, WB], 0.5)
l = layerwise_prune([WA, WB], 0.5)
print('全局剪枝各层稀疏度:', [round((w==0).mean(),2) for w in g])
print('逐层剪枝各层稀疏度:', [round((w==0).mean(),2) for w in l])
# 全局: 小权重层(B)被剪得多(因其 |w| 普遍小), 大权重层(A)被保护
assert (g[1]==0).mean() > (g[0]==0).mean(), '全局应多剪小权重层、保护大权重层'
assert abs((l[0]==0).mean() - 0.5) < 0.02, '逐层应各剪到 50%'
print('✅ 全局剪枝让各层稀疏度自适应(冗余层多剪)；逐层则均匀 —— 全局通常更优')

## 3 · 2:4 半结构稀疏：每 4 个保留 2 个

每 4 个连续权重，删掉 `|w|` 最小的 2 个、保留最大的 2 个。稀疏度固定 50%，
但规整到 NVIDIA Sparse Tensor Core 能 **2× 加速**。验证每组恰好 2 个非零。

In [ ]:
def prune_2_4(W):
    '''2:4 稀疏: 每 4 个连续元素保留 |.| 最大的 2 个。要求最后一维是 4 的倍数。'''
    shape = W.shape
    g = W.reshape(-1, 4).copy()                       # 每行 4 个一组
    for i in range(g.shape[0]):
        small_idx = np.argsort(np.abs(g[i]))[:2]      # |.| 最小的 2 个
        g[i, small_idx] = 0.0
    return g.reshape(shape)

W = rng.standard_normal((8, 16))                      # 16 是 4 的倍数
W24 = prune_2_4(W)
groups = W24.reshape(-1, 4)
nonzeros_per_group = (groups != 0).sum(axis=1)
print(f'每组非零数: 取值集合 = {set(nonzeros_per_group.tolist())} (应为 {{2}})')
print(f'整体稀疏度 = {(W24==0).mean():.2f} (2:4 固定 50%)')
assert set(nonzeros_per_group.tolist()) == {2}, '每组必须恰好 2 个非零'
assert abs((W24==0).mean() - 0.5) < 1e-9, '2:4 稀疏度应精确 50%'
# 保留的应是每组里大的
for i in range(groups.shape[0]):
    kept = np.abs(groups[i][groups[i]!=0])
    removed = np.abs(W.reshape(-1,4)[i]); removed = np.sort(removed)[:2]
    assert kept.min() >= removed.max() - 1e-9
print('✅ 2:4 稀疏正确：每组恰 2 非零、保留较大的、整体 50% —— 硬件可 2× 加速')

## 4 · 稀疏度-精度曲线

剪越狠掉越多。扫不同稀疏度，看层输出的相对误差如何随稀疏度上升。
这条曲线是剪枝的核心权衡图——告诉你能剪到多稀疏而不太掉点。

In [ ]:
d_out, d_in, n = 32, 128, 64
W = rng.standard_normal((d_out, d_in)) * 0.1
X = rng.standard_normal((d_in, n))
base = np.linalg.norm(W @ X)

print(f"{'稀疏度':>8} {'相对输出误差':>14}")
sparsities = [0.0, 0.3, 0.5, 0.7, 0.9, 0.95]
errs = []
for sp in sparsities:
    Wp, _ = magnitude_prune(W, sp)
    err = np.linalg.norm(W@X - Wp@X) / base
    errs.append(err)
    print(f'{sp:>8.2f} {err:>14.3%}')
# 误差应随稀疏度单调上升
assert all(errs[i] <= errs[i+1] + 1e-9 for i in range(len(errs)-1)), '稀疏度越高误差越大'
assert errs[0] < 1e-9, '稀疏度 0 应无误差'
print('✅ 稀疏度-精度曲线: 剪越狠误差越大。50% 通常可接受, 90%+ 需重训/SparseGPT 拯救')

## 5 · 剪枝-微调恢复

剪枝破坏了功能平衡，但**保留的权重还能动**。微调(保持掩码固定，只更新保留权重)让它们
补偿被删的，找回精度。验证：微调后层输出误差明显下降。

In [ ]:
W = rng.standard_normal((d_out, d_in)) * 0.1
X = rng.standard_normal((d_in, n))
target = W @ X                                        # 剪枝前的输出(目标)

Wp, mask = magnitude_prune(W, 0.5)
err_before = np.linalg.norm(Wp @ X - target) / np.linalg.norm(target)

# 微调: GD 更新保留权重去拟合原输出, 每步乘掩码(被剪的恒为0)
Wf = Wp.copy(); lr = 0.05
for _ in range(500):
    grad = (Wf @ X - target) @ X.T / n
    Wf = (Wf - lr * grad) * mask                      # 保持稀疏结构
err_after = np.linalg.norm(Wf @ X - target) / np.linalg.norm(target)

print(f'剪枝 50% 后 相对输出误差: 微调前 {err_before:.3%} -> 微调后 {err_after:.3%}')
assert (Wf == 0).mean() == (Wp == 0).mean(), '微调必须保持稀疏度不变'
assert err_after < err_before, '微调应降低误差'
print('✅ 微调让保留权重补偿被删的, 误差大降 —— 这是高稀疏度保精度的关键')

## 6 · SparseGPT：逆 Hessian 一次性重建

和 GPTQ 一样：剪枝是对权重的扰动，可用逆 Hessian 把误差补偿到保留权重。
用 OBS 显著性 `w²/Hinv[j,j]` 选剪谁(不只看|w|)，逐列剪+补偿。验证：MSE < 普通幅度剪枝。

In [ ]:
# 相关性激活(SparseGPT 的 Hessian 才有意义)
A = rng.standard_normal((d_in, d_in)); cov = A@A.T/d_in
Xc = cov @ rng.standard_normal((d_in, 256))
W = rng.standard_normal((d_out, d_in)) * 0.1
def out_mse(W, Wq, X): return np.mean((W@X - Wq@X)**2)

def magnitude_prune_perrow(W, sparsity):
    Wp = W.copy()
    k = int(sparsity * W.shape[1])
    for i in range(W.shape[0]):
        idx = np.argsort(np.abs(W[i]))[:k]
        Wp[i, idx] = 0.0
    return Wp

def sparsegpt(W, X, sparsity, damp=1e-2):
    d_out, d_in = W.shape
    H = X @ X.T; H = H + damp*np.mean(np.diag(H))*np.eye(d_in)
    Hinv = np.linalg.inv(H); Hd = np.diag(Hinv)
    Wq = W.copy().astype(float)
    n_prune = int(sparsity * d_in)
    for i in range(d_out):
        saliency = Wq[i]**2 / Hd                       # OBS 显著性(越小越该剪)
        prune_idx = set(np.argsort(saliency)[:n_prune].tolist())
        for j in range(d_in):                          # 逐列剪+补偿
            if j in prune_idx:
                err = Wq[i, j] / Hd[j]
                Wq[i, j] = 0.0
                if j + 1 < d_in:
                    Wq[i, j+1:] -= err * Hinv[j, j+1:]
    return Wq

mag_mse = out_mse(W, magnitude_prune_perrow(W, 0.5), Xc)
sgpt_mse = out_mse(W, sparsegpt(W, Xc, 0.5), Xc)
print(f'50% 稀疏: 幅度剪枝 MSE = {mag_mse:.4e}')
print(f'50% 稀疏: SparseGPT MSE = {sgpt_mse:.4e}  ({sgpt_mse/mag_mse:.0%} of 幅度剪枝)')
assert sgpt_mse < mag_mse, 'SparseGPT 应优于幅度剪枝'
print('✅ SparseGPT 一次性重建(无重训)优于幅度剪枝 —— 正如 GPTQ 优于 RTN(同一套逆 Hessian)')

---
## ✏️ 练习 1：全局幅度剪枝

实现 `global_magnitude_prune(weights, sparsity)`：把多个层的权重拼起来定一个统一阈值，
返回剪枝后的层列表。目标：整体稀疏度命中，各层稀疏度自适应。

In [ ]:
def global_magnitude_prune(weights, sparsity):
    # TODO:
    #   all_w = 所有层 |w| 拼成一维
    #   k = round(sparsity * all_w.size); thresh = 第 k 小的 |w| (np.partition)
    #   返回 [w * (|w| > thresh) for w in weights]
    raise NotImplementedError

In [ ]:
# —— 练习 1 自测 ——
WA = rng.standard_normal((16, 32)) * 1.0
WB = rng.standard_normal((16, 32)) * 0.05
pruned = global_magnitude_prune([WA, WB], 0.5)
total_sp = sum((w==0).sum() for w in pruned) / sum(w.size for w in pruned)
assert abs(total_sp - 0.5) < 0.02, f'整体稀疏度应≈0.5, 实际 {total_sp}'
assert (pruned[1]==0).mean() > (pruned[0]==0).mean(), '小权重层应被多剪'
print(f'✅ 练习 1 通过：全局剪枝整体稀疏度 {total_sp:.2f}，小权重层自适应多剪')

## ✏️ 练习 2：2:4 稀疏

实现 `my_prune_2_4(W)`：每 4 个连续权重保留 `|w|` 最大的 2 个。
目标：每组恰好 2 个非零、整体 50% 稀疏。

In [ ]:
def my_prune_2_4(W):
    # TODO: reshape(-1,4); 每组把 |.| 最小的 2 个置 0; reshape 回原形
    raise NotImplementedError

In [ ]:
# —— 练习 2 自测 ——
W = rng.standard_normal((8, 16))
W24 = my_prune_2_4(W)
groups = W24.reshape(-1, 4)
assert set((groups != 0).sum(axis=1).tolist()) == {2}, '每组必须恰 2 非零'
assert abs((W24==0).mean() - 0.5) < 1e-9, '应 50% 稀疏'
print('✅ 练习 2 通过：2:4 稀疏正确（每组 2 非零、整体 50%）')

## ✏️ 练习 3：稀疏度-精度曲线

实现 `sparsity_accuracy_curve(W, X, sparsities)`：对每个稀疏度做幅度剪枝，返回相对输出误差列表。
目标：误差随稀疏度单调上升。

In [ ]:
def sparsity_accuracy_curve(W, X, sparsities):
    # TODO: 对每个 sp 做 magnitude_prune(W, sp), 算 ‖Wx-Ŵx‖/‖Wx‖, 返回 list
    raise NotImplementedError

In [ ]:
# —— 练习 3 自测 ——
W = rng.standard_normal((32, 128)) * 0.1
X = rng.standard_normal((128, 64))
sps = [0.0, 0.3, 0.5, 0.7, 0.9]
errs = sparsity_accuracy_curve(W, X, sps)
assert len(errs) == len(sps)
assert all(errs[i] <= errs[i+1] + 1e-9 for i in range(len(errs)-1)), '应单调上升'
assert errs[0] < 1e-9, '稀疏度0 应无误差'
for sp, e in zip(sps, errs):
    print(f'  稀疏度 {sp:.1f}: 相对误差 {e:.3%}')
print('✅ 练习 3 通过：稀疏度-精度曲线单调上升')

## ✏️ 练习 4：SparseGPT 单行重建

实现 `sparsegpt_row(w_row, Hinv, sparsity)`：对一行权重，用 OBS 显著性 `w²/Hinv[j,j]` 选剪谁，
逐列剪+逆 Hessian 补偿到保留列。返回剪枝后的行。目标：输出误差 < 同稀疏度的幅度剪枝。

In [ ]:
def sparsegpt_row(w_row, Hinv, sparsity):
    '''对单行权重做 SparseGPT 重建。w_row:(d_in,), Hinv:(d_in,d_in)。'''
    # TODO:
    #   Hd = diag(Hinv); w = w_row.copy()
    #   n_prune = int(sparsity*len(w)); saliency = w**2/Hd
    #   prune_idx = 显著性最小的 n_prune 个下标(集合)
    #   for j in range(len(w)): if j in prune_idx: err=w[j]/Hd[j]; w[j]=0; w[j+1:]-=err*Hinv[j,j+1:]
    #   返回 w
    raise NotImplementedError

In [ ]:
# —— 练习 4 自测 ——
d = 64
A = rng.standard_normal((d, d)); cov = A@A.T/d
Xc = cov @ rng.standard_normal((d, 256))
H = Xc@Xc.T; H = H + 1e-2*np.mean(np.diag(H))*np.eye(d); Hinv = np.linalg.inv(H)
w = rng.standard_normal(d) * 0.1
w_sgpt = sparsegpt_row(w, Hinv, 0.5)
# 同稀疏度幅度剪枝
w_mag = w.copy(); w_mag[np.argsort(np.abs(w))[:d//2]] = 0
err_sgpt = np.linalg.norm((w - w_sgpt) @ Xc)
err_mag = np.linalg.norm((w - w_mag) @ Xc)
assert abs((w_sgpt==0).mean() - 0.5) < 0.05, '应约 50% 稀疏'
assert err_sgpt < err_mag, f'SparseGPT 应优于幅度剪枝: {err_sgpt:.3e} vs {err_mag:.3e}'
print(f'✅ 练习 4 通过：SparseGPT 单行重建 误差 {err_sgpt:.3e} < 幅度剪枝 {err_mag:.3e}')

---
### 📖 参考答案（先自己做，再对照）

In [ ]:
# 练习 1 参考答案
def global_magnitude_prune(weights, sparsity):
    all_w = np.concatenate([np.abs(w).ravel() for w in weights])
    k = int(round(sparsity * all_w.size))
    thresh = np.partition(all_w, k-1)[k-1]
    return [w * (np.abs(w) > thresh) for w in weights]

In [ ]:
# 练习 2 参考答案
def my_prune_2_4(W):
    shape = W.shape
    g = W.reshape(-1, 4).copy()
    for i in range(g.shape[0]):
        g[i, np.argsort(np.abs(g[i]))[:2]] = 0.0
    return g.reshape(shape)

In [ ]:
# 练习 3 参考答案
def sparsity_accuracy_curve(W, X, sparsities):
    base = np.linalg.norm(W @ X); errs = []
    for sp in sparsities:
        Wp, _ = magnitude_prune(W, sp)
        errs.append(np.linalg.norm(W@X - Wp@X) / base)
    return errs

In [ ]:
# 练习 4 参考答案
def sparsegpt_row(w_row, Hinv, sparsity):
    Hd = np.diag(Hinv); w = w_row.copy().astype(float)
    n_prune = int(sparsity * len(w))
    prune_idx = set(np.argsort(w**2 / Hd)[:n_prune].tolist())
    for j in range(len(w)):
        if j in prune_idx:
            err = w[j] / Hd[j]; w[j] = 0.0
            if j + 1 < len(w):
                w[j+1:] -= err * Hinv[j, j+1:]
    return w

---
## 🧪 真实数据胶囊：剪枝真实 GPT-2 权重 + 微调恢复

用**真实 GPT-2** 权重做幅度剪枝，再微调恢复，看真实权重上的稀疏-精度与恢复效果。
**联网失败自动回退**到统计匹配的合成权重，结论不变。

In [ ]:
def load_gpt2_weight():
    '''真实 GPT-2 一个 MLP 权重切片; 失败回退合成。'''
    try:
        from transformers import GPT2Model
        m = GPT2Model.from_pretrained('gpt2')
        W = m.h[0].mlp.c_proj.weight.detach().numpy().astype(np.float64)
        W = W[:64, :128]
        print(f'[真实 GPT-2] c_proj 切片 shape={W.shape}')
        return W
    except Exception as e:
        print(f'[回退合成] ({type(e).__name__}) 用统计匹配合成权重')
        rng2 = np.random.default_rng(17)
        return rng2.standard_normal((64, 128)) * 0.1

Wg = load_gpt2_weight()
Xg = rng.standard_normal((Wg.shape[1], 128))
print(f'权重统计: std={Wg.std():.4f}, 稀疏潜力(|w|<0.5std 占比)={ (np.abs(Wg)<0.5*Wg.std()).mean():.1%}')

In [ ]:
def prune_and_finetune(W, X, sparsity, ft_steps=400, lr=0.05):
    # TODO:
    #   target = W@X; Wp,mask = magnitude_prune(W,sparsity)
    #   err_before = ‖Wp@X-target‖/‖target‖
    #   微调: Wf=Wp.copy(); 重复 ft_steps: grad=(Wf@X-target)@X.T/X.shape[1]; Wf=(Wf-lr*grad)*mask
    #   err_after = ‖Wf@X-target‖/‖target‖
    #   返回 (err_before, err_after)
    raise NotImplementedError

In [ ]:
# 自测
eb, ea = prune_and_finetune(Wg, Xg, sparsity=0.5)
print(f'真实(或合成)GPT-2 权重 50% 剪枝:')
print(f'  微调前 相对输出误差 = {eb:.3%}')
print(f'  微调后 相对输出误差 = {ea:.3%}  (恢复到 {ea/eb:.0%})')
assert ea < eb, '微调应降低误差'
print('✅ 胶囊通过：真实权重上剪枝-微调有效 —— 保留权重补偿被删的，找回精度')

In [ ]:
# 📖 胶囊参考答案
def prune_and_finetune(W, X, sparsity, ft_steps=400, lr=0.05):
    target = W @ X
    Wp, mask = magnitude_prune(W, sparsity)
    err_before = np.linalg.norm(Wp@X - target) / np.linalg.norm(target)
    Wf = Wp.copy()
    for _ in range(ft_steps):
        grad = (Wf @ X - target) @ X.T / X.shape[1]
        Wf = (Wf - lr * grad) * mask
    err_after = np.linalg.norm(Wf@X - target) / np.linalg.norm(target)
    return err_before, err_after

### 小结
- **剪枝 = 把不重要权重置 0**，压的是『非零参数个数』(与量化压『每参数 bit』正交、可叠加)。
- **幅度剪枝**：删最小 |w|，简单强基线。盲点和 RTN 一样——只看权重不看对输出的影响。
- **结构**决定能否加速：非结构化最准但通用硬件**不加速**(0也得乘)；结构化加速但掉点大；**2:4** 是甜点。
- **2:4 稀疏**：每 4 留 2，50% 稀疏，Sparse Tensor Core 原生 **2× 加速** —— 最实用的稀疏格式。
- **全局剪枝**让各层稀疏度自适应(冗余层多剪)，通常优于逐层。
- **剪枝-微调/IMP**：保留权重重训补偿被删的，高稀疏度保精度的关键。
- **SparseGPT** = GPTQ 搬到剪枝：逆 Hessian 一次性重建，无需重训，优于幅度剪枝(同一套数学)。
- **彩票假说**：稠密网含可用原初始化单独训练的稀疏子网 —— 稀疏性是网络的内在属性。

🎉 **恭喜你读完并写完了整门课**。你现在掌握了现代大模型压缩的五大支柱(量化/GPTQ·AWQ/fp8/蒸馏/剪枝)，
并理解它们同属一个『带约束的逐层近似』框架。下一步：把这些验证过的逻辑接到 bitsandbytes/AutoGPTQ/TransformerEngine 等生产栈。